# 📈 Stock Price Predictor — Linear Regression

This notebook walks through predicting the **next day's closing price** of a stock  
using historical OHLCV data and a Linear Regression model.

**Pipeline:**
1. Load data via Yahoo Finance
2. Feature engineering
3. Time-series train/test split
4. Train Linear Regression
5. Evaluate (MAE, RMSE, R²)
6. Visualise results

In [ ]:
# Install required library (run once)
!pip install yfinance scikit-learn matplotlib pandas numpy --quiet

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Libraries loaded ✅')

## Step 1 — Load Stock Data

In [ ]:
TICKER = 'AAPL'
START  = '2020-01-01'
END    = '2024-01-01'

stock = yf.download(TICKER, start=START, end=END, auto_adjust=True)
print(f'Shape: {stock.shape}')
stock.head()

## Step 2 — Feature Engineering

In [ ]:
data = stock[['Open', 'High', 'Low', 'Volume', 'Close']].copy()

# Target = next day's closing price
data['Target'] = data['Close'].shift(-1)
data.dropna(inplace=True)

X = data[['Open', 'High', 'Low', 'Volume']]
y = data['Target']

print(f'Features : {X.shape[1]} columns, {X.shape[0]} rows')
data.tail()

## Step 3 — Chronological Train / Test Split (80 / 20)

In [ ]:
split = int(len(X) * 0.80)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

## Step 4 — Train the Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print('Coefficients:')
for feat, coef in zip(X.columns, model.coef_):
    print(f'  {feat:10s}: {coef:.6f}')
print(f'Intercept   : {model.intercept_:.4f}')

## Step 5 — Predictions & Evaluation

In [ ]:
predictions = model.predict(X_test)

mae  = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2   = r2_score(y_test, predictions)

print(f'MAE  : ${mae:.4f}')
print(f'RMSE : ${rmse:.4f}')
print(f'R²   : {r2:.4f}')

## Step 6 — Visualise: Actual vs Predicted

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(y_test.values,  label='Actual Price',    color='#1f77b4', linewidth=1.5)
plt.plot(predictions,    label='Predicted Price', color='#ff7f0e', linewidth=1.5, linestyle='--')
plt.title(f'{TICKER} — Actual vs Predicted Closing Price', fontsize=15, fontweight='bold')
plt.xlabel('Trading Days (Test Set)')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/AAPL_prediction.png', dpi=150)
plt.show()